# Customer Churn Prediction — Milestone 2
## Model Building, Performance Improvement, Comparison and Final Selection

**Continuation of:** *Customer Churn Prediction — Milestone 1* (the source notebook supplied as `Customer_Churn_Prediction MS2.ipynb`, Sections 1–3 before Model Development).

This notebook begins with a concise recap of Milestone 1, independently reloads the original data, recreates the exact cleaning, split and preprocessing definitions from Milestone 1, then proceeds directly to model building. No EDA or earlier narrative is repeated.

## Milestone 1 recap and Milestone 2 handoff

**Reference:** `Customer_Churn_Prediction MS2.ipynb` (Milestone 1 analysis). Milestone 1 examined 11,260 customers, 19 fields and approximately 16.8% churn; it included cleaning, feature exploration, an 80:20 stratified split and a Streamlit application.

This standalone Milestone 2 notebook reloads the source data and imports preprocessing from `scripts/preprocessing.py`, without repeating EDA. Milestone 2 compares and tunes models, selects the best model, and connects it to ChurnGuard automated batch prediction and the existing Streamlit application. The fitted pipeline must be validated with both consumers before deployment.


**Standalone setup:** Keep `scripts/preprocessing.py` beside the notebook under the project root. Run Jupyter with the repository root as its working directory. Install dependencies from your project environment. The model artifact stores the shared transformer by module reference; deploy the same `scripts` package with Streamlit and ChurnGuard.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, RandomizedSearchCV, ParameterGrid
from sklearn.metrics import (ConfusionMatrixDisplay, confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score, average_precision_score)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import json, joblib
from IPython.display import display
from xgboost import XGBClassifier

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
# Original Milestone 1 worksheet; change DATA_FILE if your workbook lives elsewhere.
DATA_FILE = Path("data/Customer Churn Data.xlsx")
if not DATA_FILE.exists():
    alternative = Path("Customer Churn Data.xlsx")
    if alternative.exists():
        DATA_FILE = alternative
    else:
        raise FileNotFoundError(
            "Original dataset is required to run this standalone notebook. "
            "Place 'Customer Churn Data.xlsx' in ./data/ or edit DATA_FILE above. "
            "It was not included with the supplied notebook."
        )

# Reconstruct the Milestone 1 handoff without repeating EDA.
from scripts.preprocessing import build_preprocessor

df = pd.read_excel(DATA_FILE, sheet_name="Data for DSBA")
df.columns = df.columns.str.strip().str.lower()
TARGET_COL, id_cols = "churn", "accountid"
X = df.drop(columns=[TARGET_COL, id_cols])
y = df[TARGET_COL]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
train_accounts = df.loc[X_train.index, id_cols].copy()
test_accounts = df.loc[X_test.index, id_cols].copy()
preprocessor = build_preprocessor(X_train)
assert set(X_train.index).isdisjoint(X_test.index)
print(f"Dataset: ***/data/{DATA_FILE.name} | Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Features: {X_train.shape[1]} | Churn: {y_train.mean():.2%} train, {y_test.mean():.2%} test")
print("Shared preprocessing module ready (unfitted).")


**Milestone 1 continuity and deployment note:** This notebook reconstructs the data handoff and preprocessing from Milestone 1, then starts model building. The automated churn-prediction application was already developed; this milestone benchmarks and tunes additional algorithms. Replacing the deployed model requires exporting the final fitted preprocessing-and-classifier pipeline and validating that the application uses the same feature schema. No deployment or prediction results are claimed until tested.

## 1. Model Building and Evaluation (5 points)

### 1.1 Models and metric choice

| Model | Purpose | Origin |
|---|---|---|
| Logistic Regression | Linear baseline | Milestone 1 |
| Decision Tree | Nonlinear rules | Milestone 1 |
| Random Forest | Bagging ensemble | Milestone 1 |
| Gradient Boosting | Sequential boosting | Milestone 1 |
| XGBoost | Regularized boosting | Milestone 1 |
| KNN | Distance-based comparison | New |
| Gaussian Naive Bayes | Probabilistic baseline | New |
| SVM (RBF) | Kernel classifier | New |
| Extra Trees | Randomized ensemble | New |

**F1** is the primary selection metric due to churn imbalance (~16.8%), supported by precision, recall, PR-AUC, ROC-AUC and confusion matrices. Preprocessing is fitted within each training fold.


<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Helper Function

In [ ]:
# Train models using identical preprocessing.
def train_models(models, preprocessor, X_train, y_train):
    """Fit each model with a fresh copy of the shared preprocessing pipeline.
       Displays training progress, and returns a dictionary of trained models for evaluation and comparison.
    """
    trained = {}
    for name, model in models.items():
        # Keep preprocessing and classifier instances independent for each model.
        pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("classifier", clone(model)),
        ])
        trained[name] = pipeline.fit(X_train, y_train)
        print(f"{name}: Training completed")
    return trained

In [ ]:
# Evaluate the selected model on the held-out test set ONLY after model selection.
def evaluate_final_model(name, model, X_test, y_test):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    result = pd.DataFrame([{
        "Model": name, "Dataset": "Test",
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-Score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob),
        "PR-AUC": average_precision_score(y_test, prob),
    }])
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, labels=[0, 1], display_labels=["Retained", "Churned"],
        cmap="Blues", colorbar=False
    )
    plt.title(f"Final held-out test: {name}")
    plt.show()
    return result

In [ ]:
# Tune each model with stratified cross-validation and rank by validation F1.

def tune_models(models, tuning_params, preprocessor, X_train, y_train):
    """Return the best fitted model and CV summary for each classifier."""
    # Preserve churn proportions in each fold.
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    tuned_models, results = {}, []

    for name, model in models.items():
        print(f"\nTuning {name}...")

        # Clone preprocessing and classifier so each search is independent.
        pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("classifier", clone(model)),
        ])

        # Search parameter combinations using average precision (F1).
        search = RandomizedSearchCV(
            pipeline,
            tuning_params[name],
            n_iter=min(20, len(ParameterGrid(tuning_params[name]))),
            scoring="f1",
            cv=cv,
            n_jobs=-1,
            random_state=42,
            verbose=1,
            refit=True,
        )
        search.fit(X_train, y_train)
        tuned_models[name] = search.best_estimator_
        results.append({
            "Model": name,
            "CV F1": search.best_score_,
            "Best Parameters": search.best_params_,
        })

        print(f"Best parameters: {search.best_params_}")
        print(f"Best CV F1: {search.best_score_:.3f}")

    # Summarize models from highest to lowest cross-validation F1.
    summary = (
        pd.DataFrame(results)
        .sort_values("CV F1", ascending=False)
        .set_index("Model")
    )
    display(summary.round(3))
    return tuned_models, summary

In [ ]:
# Compare models with stratified CV and plot out-of-fold confusion matrices.

def evaluate_cv(models, preprocessor, X_train, y_train):
    """Rank models by 5-fold PR-AUC and show their out-of-fold confusion matrices."""
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scoring = {
        "Precision": "precision", "Recall": "recall", "F1": "f1",
        "ROC-AUC": "roc_auc", "PR-AUC": "average_precision",
    }
    results, matrices = [], {}

    for name, model in models.items():
        # Use existing pipelines as-is; otherwise add shared preprocessing.
        pipeline = clone(model) if isinstance(model, Pipeline) else Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("classifier", clone(model)),
        ])

        scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
        # Out-of-fold predictions keep each row unseen during its prediction.
        pred = cross_val_predict(pipeline, X_train, y_train, cv=cv, n_jobs=-1)
        results.append({
            "Model": name,
            **{metric: scores[f"test_{metric}"].mean() for metric in scoring},
            "PR-AUC SD": scores["test_PR-AUC"].std(),
        })
        matrices[name] = confusion_matrix(y_train, pred, labels=[0, 1])

    # Sort models by mean cross-validation PR-AUC.
    cv_df = pd.DataFrame(results).sort_values("F1", ascending=False).set_index("Model")
    display(cv_df.T.round(4))

    fig, axes = plt.subplots(1, len(cv_df), figsize=(4 * len(cv_df), 4), squeeze=False)
    for ax, name in zip(axes.ravel(), cv_df.index):
        ConfusionMatrixDisplay(
            matrices[name], display_labels=["Retained", "Churned"]
        ).plot(ax=ax, cmap="Blues", colorbar=False)
        ax.set_title(name)
    plt.tight_layout()
    plt.show()
    return cv_df

### 1.2 Configure nine baseline models

In [ ]:
# Six complementary model families; churn weighting is computed from training labels only.
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=1),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "SVM (RBF)": SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=1),
    "XGBoost": XGBClassifier(n_estimators=200, scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42, n_jobs=1),
}


In [ ]:
# Train Baseline Models
trained_models = train_models(models, preprocessor, X_train, y_train)

# Summarize trained models.
model_summary = pd.DataFrame({
    "Model": list(trained_models),
    "Training Records": len(X_train),
    "Features": X_train.shape[1],
    "Status": "Trained"
})

display(model_summary)

### 1.3 Baseline training overview

### 1.4 Baseline performance — five-fold stratified cross-validation

In [ ]:
# Evaluate baseline models using only the training set (no test-set inspection).
baseline_cv_results = evaluate_cv(models, preprocessor, X_train, y_train)
display(baseline_cv_results.round(4))

**Interpretation guide:** Rank baseline models by CV F1; inspect recall, precision, PR-AUC and fold variability. Complete the narrative from executed scores, not assumed results.

## 2. Model Improvement, Comparison and Final Selection (15 points)

All nine models undergo hyperparameter tuning using 5-fold stratified cross-validation, with F1 as the primary selection metric. The highest cross-validation score may be optimistic due to model selection. The untouched test set will therefore be used only to evaluate the final selected model.

### 2.1 Hyperparameter search spaces — six models

In [ ]:
tuning_params = {
 "Logistic Regression": {"classifier__C": [0.01,0.1,1,10], "classifier__class_weight": [None,"balanced"]},
 "Decision Tree": {"classifier__max_depth": [None, 5, 10, 20], "classifier__min_samples_leaf": [1, 3, 5]},
 "Gradient Boosting": {"classifier__n_estimators": [100, 200], "classifier__learning_rate": [0.03, 0.1], "classifier__max_depth": [2, 3]},
 "Extra Trees": {"classifier__n_estimators": [100, 200], "classifier__max_depth": [None, 10, 20], "classifier__min_samples_leaf": [1, 3]},
 "KNN": {"classifier__n_neighbors": [3,5,9,15,25], "classifier__weights": ["uniform","distance"], "classifier__p": [1,2]},
 "Naive Bayes": {"classifier__var_smoothing": [1e-11,1e-10,1e-9,1e-8,1e-7]},
 "SVM (RBF)": {"classifier__C": [0.1,1,10], "classifier__gamma": ["scale",0.01,0.1]},
 "Random Forest": {"classifier__n_estimators": [100,200,300], "classifier__max_depth": [None,8,15], "classifier__min_samples_leaf": [1,3,5]},
 "XGBoost": {"classifier__n_estimators": [100,200,350], "classifier__max_depth": [3,5,7], "classifier__learning_rate": [0.03,0.1], "classifier__subsample": [0.8,1.0], "classifier__colsample_bytree": [0.8,1.0]},
}


In [ ]:
# Train tuned Models
tuned_models, tuning_results = tune_models(
    models, tuning_params, preprocessor, X_train, y_train
)

In [ ]:
# Cross-validated metrics for each tuned model (training data only).
# Note: these CV scores are descriptive; they are not nested-CV unbiased estimates.
tuned_cv_results = evaluate_cv(tuned_models, preprocessor, X_train, y_train)
display(tuned_cv_results.round(4))

**Tuned-model commentary:** Compare CV F1 with the baseline and discuss recall/precision trade-offs. Search-selected CV scores can be optimistic; do not treat them as independent test estimates.

### 2.2 Baseline vs tuned comparison (training data only)

In [ ]:
# Compare baseline and tuned cross-validation results without using test data.
metrics = ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC", "PR-AUC SD"]
comparison = pd.concat({
    "Baseline CV": baseline_cv_results[metrics],
    "Tuned CV": tuned_cv_results[metrics]
}, axis=1)
comparison[("Change", "F1")] = tuned_cv_results["F1"] - baseline_cv_results["F1"]
comparison = comparison.sort_values(("Tuned CV", "F1"), ascending=False)
display(comparison.round(4))

### 2.3 Final model selection by tuning CV F1

**Selection rule:** Highest mean five-fold CV F1 from hyperparameter search; secondary consideration: recall, PR-AUC, variance and deployment compatibility. The deployed XGBoost model is not automatically the winner.

The test set has not been used for model or threshold selection. If the winner is not XGBoost, preserve the existing ChurnGuard deployment until its prediction script and schema are migrated and validated.

In [ ]:
# Rank Tuned Models by Cross-Validation PR-AUC

cv_comparison = tuning_results[["CV F1"]].sort_values(
    "CV F1", ascending=False
)

display(cv_comparison.round(4))

# Select Best Model
best_model_name = cv_comparison.index[0]
best_model = tuned_models[best_model_name]

print(f"\nSelected Model: {best_model_name}")
print(f"CV F1: {cv_comparison.iloc[0, 0]:.4f}")

### 2.4 Final held-out test evaluation (selected model only)

This is the only candidate tested on the untouched holdout. Report F1, precision, recall, PR-AUC, ROC-AUC and the confusion matrix, with a short business interpretation.

In [ ]:
# Run once after model selection. No other candidate is evaluated on the test set.
final_results = evaluate_final_model(best_model_name, best_model, X_test, y_test)
display(final_results.set_index(["Model", "Dataset"]).round(4))

**Complete after execution:** State the selected model, CV F1, final test F1, recall, precision and PR-AUC. Comment on false negatives (missed churners), false positives (unnecessary interventions) and any validation-to-test gap. Do not claim improvement without observed scores.

### 2.6 Conclusion and rationale
After executing the notebook, report (1) baseline CV PR-AUC and secondary metrics for all six models, (2) tuning settings and tuned CV scores, (3) comparison and final selection based on CV PR-AUC, and (4) the selected model’s **single** held-out test evaluation. Discuss any precision–recall trade-off and whether the test score is consistent with cross-validation.

**Note:** CV scores computed after selecting hyperparameters on the same folds can be optimistic. For a more rigorous unbiased comparison, use nested cross-validation; the final untouched test evaluation remains the primary generalization check.

## 3. Business interpretation and ChurnGuard deployment continuity

The model comparison is an **experiment**, not an instruction to overwrite production artifacts. ChurnGuard already performs automated batch predictions on new CSV/XLSX files using `predict_changed.py` and GitHub Actions. Its existing `churnguard_xgboost.joblib` and `feature_schema.json` are the deployment contract, and batch input also requires `accountid` for output identification.

The risk bands **0.30 / 0.60 / 0.80** are existing operational cutoffs, not validated optima. Keep them labeled as business-defined until validated on training/validation data. Compare the selected model with the current production XGBoost model and migrate only after schema and output compatibility checks.


In [ ]:
# Deployment compatibility review — no automatic overwrite of existing artifacts.
from pathlib import Path
schema_path = Path("feature_schema.json")
if schema_path.exists():
    with schema_path.open(encoding="utf-8") as f:
        deployed_schema = json.load(f)
    print("Existing feature_schema.json found. Review its expected feature list against:")
    print(list(X_train.columns))
    print("Schema content:", deployed_schema)
else:
    print("No feature_schema.json in this working directory; check repository root.")
print("Final model:", best_model_name)
print("Current ChurnGuard artifact: churnguard_xgboost.joblib")
print("Deployment ready without script migration:", best_model_name == "XGBoost")


### 3.1 Export candidate (do not overwrite production automatically)

Save a candidate pipeline for inspection. Before replacing `churnguard_xgboost.joblib`, verify the pipeline input columns, `predict_proba` output, customer identifier handling and `predict_changed.py` integration on a sample file. If a non-XGBoost model wins, update the automation to use a model-agnostic artifact name and test it before deployment.


In [ ]:
# Candidate model includes fitted preprocessing. Existing production artifact is untouched.
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
candidate_path = output_dir / "churnguard_candidate_pipeline.joblib"
joblib.dump(best_model, candidate_path)
feature_schema_candidate = {
    "feature_columns": list(X_train.columns),
    "identifier_column": "accountid",
    "target_column": "churn",
    "selected_model": best_model_name,
    "model_artifact": str(candidate_path),
    "risk_thresholds": [0.30, 0.60, 0.80],
    "risk_threshold_status": "business_defined_not_optimized",
}
with (output_dir / "feature_schema_candidate.json").open("w", encoding="utf-8") as f:
    json.dump(feature_schema_candidate, f, indent=2)
print("Candidate exported:", candidate_path)
print("Production files not overwritten.")


### 3.2 Batch-scoring smoke test

This verifies the candidate's prediction interface and input schema. It does **not** prove GitHub Actions compatibility; run `predict_changed.py` against a representative input in a test branch before replacing production artifacts.


In [ ]:
sample = X_test.head(5).copy()
assert list(sample.columns) == list(X_train.columns)
probs = best_model.predict_proba(sample)[:, 1]
assert len(probs) == len(sample) and np.isfinite(probs).all()
preview = pd.DataFrame({"accountid": test_accounts.loc[sample.index].values,
                        "churn_probability": probs,
                        "predicted_churn": (probs >= 0.5).astype(int)})
display(preview)
